# 03 · The full graph: from transaction to decision

**You'll learn:** how `graph.py` wires the nodes together, what happens on each path (approve, decline, investigate), how the run pauses for a human, how the decision resumes it, and how to read the audit trail.

**Time:** about 10 minutes · **Cost:** free tier or about 1¢ · **Needs:** an API key in `.env`.

```mermaid
flowchart LR
  I["ingest"] --> T{"triage<br/>rules, no LLM"}
  T -->|"score < 0.30"| AA["auto_approve"]
  T -->|"score ≥ 0.65"| AD["auto_decline"]
  T -->|"in between"| TA["transaction_analyst"]
  T -->|"in between"| IV["identity_verifier"]
  T -->|"in between"| PD["pattern_detector"]
  TA --> AG["aggregator"]
  IV --> AG
  PD --> AG
  AG -->|"composite ≥ 0.5"| HR["human_review<br/>pauses here"]
  AG -->|"below 0.5"| R["resolution"]
  HR --> R
  AA --> R
  AD --> R
  R --> E(["END"])
```

| Part | What it covers | Code |
|---|---|---|
| 1 | Setup | |
| 2 | The wiring | `graph.py: build_graph()` |
| 3 | The fast paths: no LLM | |
| 4 | The investigate path, paused for review | |
| 5 | The human decision | `ui/logic.py: resume_transaction()` |
| 6 | Any transaction ID | |
| 7 | Recap and things to try | |

---
## Part 1 · Setup

### Setup
The next cell connects the notebook to the project. It gives this notebook **its own** database files in `.notebook_state/`, so nothing here touches the Streamlit app's data. It prints which LLM this run uses and whether its key is set (never the key itself). Set `SHOW_LIVE_LOGS = True` to see each node's live log line.

Importing `graph` builds the real compiled graph with a SQLite checkpointer and store, pointed at this notebook's own files.

In [ ]:
# ── Settings ────────────────────────────────────────────────────────────
SHOW_LIVE_LOGS = False  # True = print every node's live log line, like the terminal does

# Setup: run from the repo root, and give this notebook its own database files in
# .notebook_state/ so it never touches the app's checkpoints.sqlite / account_memory.sqlite.
import logging
import os
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
STATE_DIR = ROOT / ".notebook_state"
STATE_DIR.mkdir(exist_ok=True)
os.environ["CHECKPOINT_DB_PATH"] = str(STATE_DIR / "03_checkpoints.sqlite")
os.environ["MEMORY_DB_PATH"] = str(STATE_DIR / "03_memory.sqlite")
os.environ.setdefault("FASTMCP_LOG_LEVEL", "WARNING")  # quiet the MCP server subprocess
warnings.filterwarnings("ignore")

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")  # never overrides the two database paths set above

import pandas as pd

import core.audit  # noqa: F401  (creates the "fraud_guard" live logger)
from core.llm import llm_model, llm_provider

logging.getLogger("fraud_guard").setLevel(logging.INFO if SHOW_LIVE_LOGS else logging.WARNING)
pd.set_option("display.max_colwidth", 160)
key = {"gemini": "GEMINI_API_KEY", "openai": "OPENAI_API_KEY"}.get(llm_provider(), "?")
print(f"LLM: {llm_provider()} / {llm_model()}   ({key} set: {'yes' if os.environ.get(key) else 'NO'})")

import uuid

from IPython.display import Markdown

from core.state import new_state
from data.synthetic_transactions import SCENARIOS, get_scenario
from graph import build_graph, graph


def new_thread(name: str) -> dict:
    # One thread_id per run, like the app: the checkpointer keys saved state by it.
    return {"configurable": {"thread_id": f"{name}-{uuid.uuid4().hex[:6]}"}}


def audit_table(values: dict) -> pd.DataFrame:
    log = pd.DataFrame(values["audit_log"])
    log["detail"] = log.apply(lambda r: r.get("text") if isinstance(r.get("text"), str)
                              else f"{r.get('tool')}({r.get('args') or ''}) {r.get('result') or ''}", axis=1)
    return log[["node", "kind", "detail"]]

---
## Part 2 · The wiring

`build_graph()` is the whole pipeline in about 30 lines: add the 10 nodes, connect them with edges, add the two routers as conditional edges, and compile with a checkpointer, a store and `interrupt_before=["human_review"]`. This is the real code:

In [ ]:
import inspect

print(inspect.getsource(build_graph))

LangGraph can draw the compiled graph itself. Dotted arrows are conditional edges; `human_review` is marked as an interrupt.

In [ ]:
from langchain_core.runnables.graph import NodeStyles

# Explicit colours so the boxes read on both light and dark themes
# (LangGraph's default fills are pale but leave the text colour to the theme).
GRAPH_STYLES = NodeStyles(
    default="fill:#DCE6FA,stroke:#2459C2,stroke-width:1px,color:#15202E,line-height:1.2",
    first="fill:#2459C2,stroke:#2459C2,color:#FFFFFF",
    last="fill:#2F7A4A,stroke:#2F7A4A,color:#FFFFFF",
)


Markdown("```mermaid\n" + graph.get_graph().draw_mermaid(node_colors=GRAPH_STYLES) + "\n```")

---
## Part 3 · The fast paths: no LLM

Most traffic ends here. A $6.50 coffee is approved and a US-to-Nigeria purchase 10 minutes after a US one is declined, both by triage alone, in milliseconds.

In [ ]:
import time

for name in ("clean_coffee", "impossible_travel"):
    start = time.time()
    out = graph.invoke(new_state(SCENARIOS[name]), new_thread(name))
    print(f"{name}: triage {out['triage_score']:.2f} -> {out['triage_decision']} -> {out['final_decision']} ({(time.time() - start) * 1000:.0f} ms)")
    display(audit_table(out))

---
## Part 4 · The investigate path, paused for review

The $18,000 wire scores in the middle band, so triage fans out to the three specialists **in parallel**, the aggregator scores their findings, and if the composite score is 0.5 or more the graph **stops before `human_review`**. `invoke()` returns, and `get_state()` shows where it is waiting.

In [ ]:
wire_config = new_thread("borderline_wire")
start = time.time()
graph.invoke(new_state(SCENARIOS["borderline_wire"]), wire_config)
snapshot = graph.get_state(wire_config)

print(f"took {time.time() - start:.1f}s · waiting before: {snapshot.next}")
print(f"triage {snapshot.values['triage_score']:.2f} · composite {snapshot.values['composite_score']:.2f}")
print("aggregator reasoning:", snapshot.values["aggregate_reasoning"])

The audit trail shows every step so far, including each specialist's tool calls. The three specialists' entries interleave because they ran at the same time:

In [ ]:
audit_table(snapshot.values)

---
## Part 5 · The human decision

This is what the Streamlit review queue and `cli.py review` do (`ui/logic.py: resume_transaction()`): write the reviewer's choice into the saved state, then continue the same thread. The choices are `approve`, `decline` or `escalate`.

In [ ]:
graph.update_state(wire_config, {"human_decision": "approve"})
final = graph.invoke(None, wire_config)

print("final decision:", final["final_decision"])
audit_table(final).tail(3)

---
## Part 6 · Any transaction ID

Type any ID; it becomes a reproducible synthetic transaction (notebook 01, Part 2). If it pauses for review, this cell declines it.

In [ ]:
MY_ID = "conference-demo-7"

config = new_thread(MY_ID)
out = graph.invoke(new_state(get_scenario(MY_ID)), config)
if graph.get_state(config).next:
    graph.update_state(config, {"human_decision": "decline"})
    out = graph.invoke(None, config)
print(f"{MY_ID}: ${out['transaction']['amount']:,.2f} · triage {out['triage_score']:.2f} ({out['triage_decision']}) · final: {out['final_decision']}")

---
## Part 7 · Recap and things to try

**What happened:** one graph handled every path. Cheap rules settled the obvious cases; the ambiguous wire fanned out to three agents in parallel, was scored, and **paused**; a human's decision resumed the same thread to a final, audited outcome.

**Try this**
- Resume a paused run with `"escalate"` instead and check `final_decision`.
- Run `borderline_restart_demo` and read the analyst's sanctions finding in the audit table.
- Try a few IDs in Part 6 until one pauses for review.

**Next:** `04_persistence_and_memory.ipynb`: kill the process mid-review and pick up where it left off.